# Lab 1: Introduction to Apache Spark (PySpark)

Welcome to the first lab! In this lab, you will learn the basics of Apache Spark using PySpark, Spark's Python API.
We will be working with a small dataset of movies (`movies.csv`).

**Learning Objectives:**
- Initialize a `SparkSession`.
- Read data from a CSV file into a Spark DataFrame.
- Perform basic Data Explorations (`show`, `printSchema`, `count`).
- Perform basic Transformations (`select`, `filter`).
- Perform complex Transformations (`split`, `explode`, `groupBy`, `avg`).
- Write results back to storage.

## 1. Initialize SparkSession
The `SparkSession` is the entry point into all functionality in Spark. When running this in Dataproc Jupyter, a `SparkSession` is usually pre-configured and available as `spark`. Let's create one explicitly just in case.

In [ ]:
from pyspark.sql import SparkSession

# Create or get the existing SparkSession
spark = SparkSession.builder \
    .appName("Lab 1 - Intro to PySpark - Movies") \
    .getOrCreate()

# Set log level to WARN to avoid excessive INFO messages
spark.sparkContext.setLogLevel("WARN")

print("SparkSession initialized!\n")
spark

## 2. Load the Dataset
Let's read the `movies.csv` file. In a real Dataproc cluster, this path might point to a Google Cloud Storage bucket (`gs://bucket-name/movies.csv`). For this lab, we'll assume it's in the same directory.

In [ ]:
file_path = "movies.csv"

# Read the CSV file into a DataFrame
df_movies = spark.read.csv(file_path, header=True, inferSchema=True)

print("Data loaded successfully!")

## 3. Explore the Data
Before performing transformations, it's good practice to understand the structure of the data.

In [ ]:
# Print the schema (column names and data types)
print("--- Schema of the DataFrame ---")
df_movies.printSchema()

# Show the first 5 rows (truncate=False ensures columns aren't cut off)
print("\n--- First 5 rows ---")
df_movies.show(5, truncate=False)

# Count the total number of records
print(f"\nTotal number of movies: {df_movies.count()}")

## 4. Basic Transformations
Let's try filtering the data and selecting specific columns.

In [ ]:
from pyspark.sql.functions import col, desc

print("--- Movies released after 2000 ---")
df_modern_movies = df_movies.filter(col("year") >= 2000)
df_modern_movies.show(5, truncate=False)

print("--- Top 5 Highest Rated Movies ---")
df_top_rated = df_movies.select("title", "year", "rating") \
                        .orderBy(desc("rating"))
df_top_rated.show(5, truncate=False)

## 5. Complex Transformations (Explode and GroupBy)
The `genres` column contains multiple genres separated by a pipe `|`. We want to find the average rating per individual genre.

In [ ]:
from pyspark.sql.functions import explode, split, avg

# Step 1: Split the string into an array, then explode the array into multiple rows
df_exploded_genres = df_movies.withColumn("genre", explode(split(col("genres"), r"\|")))

# Step 2: Group by the new genre column and calculate average rating
df_genre_ratings = df_exploded_genres.groupBy("genre") \
    .agg(avg("rating").alias("avg_rating")) \
    .orderBy(desc("avg_rating"))

print("--- Average Rating by Genre ---")
df_genre_ratings.show(truncate=False)

## 6. Write Data (Action)
Finally, we can write the transformed data back out to storage. In Dataproc, this would typically go to GCS.

In [ ]:
output_path = "output/genre_ratings"

# We use repartition(1) here to output a single file, since the dataset is tiny.
# Mode 'overwrite' allows running this cell multiple times.
df_genre_ratings.repartition(1).write.csv(output_path, header=True, mode="overwrite")

print(f"Data successfully written to {output_path}")

In [ ]:
from pyspark.sql import SparkSession

# Create or get the existing SparkSession
spark = SparkSession.builder \
    .appName("Lab 1 - Intro to PySpark - Movies") \
    .getOrCreate()
file_path = "data/movies.csv"
df = spark.read.csv(file_path, header=True, inferSchema=True)


In [ ]:
df_modern_movies = df_movies.filter(col("year") >= 2000)

In [ ]:
output_path = "output/filtered"

# We use repartition(1) here to output a single file, since the dataset is tiny.
# Mode 'overwrite' allows running this cell multiple times.
df_modern_movies.write.csv(output_path, header=True, mode="overwrite")

In [ ]:
from pyspark.sql.functions import explode, split, avg

output_path = "output/aggregated"

# Step 1: Split the string into an array, then explode the array into multiple rows
df_exploded_genres = df_movies.withColumn("genre", explode(split(col("genres"), r"\|")))

# Step 2: Group by the new genre column and calculate average rating
df_genre_ratings = df_exploded_genres.groupBy("genre") \
    .agg(avg("rating").alias("avg_rating")) \
    .write.csv(output_path, header=True, mode="overwrite")